In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [19]:
# Load the data
df = pd.read_csv('csgo_round_snapshots.csv')


In [20]:
# Convert the Target (y) to 1s and 0s
df['round_winner'] = df['round_winner'].map({'CT': 0, 'T': 1})

In [21]:
# Convert True/False to 1/0
df['bomb_planted'] = df['bomb_planted'].astype(int)

In [22]:
# Handle Map Names (One-Hot Encoding)
df = pd.get_dummies(df, columns=['map'])

In [23]:
print("Data shape after cleaning:", df.shape)
df.head()

Data shape after cleaning: (122410, 104)


,time_left,ct_score,t_score,bomb_planted,ct_health,t_health,ct_armor,t_armor,ct_money,t_money,...,t_grenade_decoygrenade,round_winner,map_de_cache,map_de_dust2,map_de_inferno,map_de_mirage,map_de_nuke,map_de_overpass,map_de_train,map_de_vertigo
0,175.00,0.0,0.0,0,500.0,500.0,0.0,0.0,4000.0,4000.0,...,0.0,0,False,True,False,False,False,False,False,False
1,156.03,0.0,0.0,0,500.0,500.0,400.0,300.0,600.0,650.0,...,0.0,0,False,True,False,False,False,False,False,False
2,96.03,0.0,0.0,0,391.0,400.0,294.0,200.0,750.0,500.0,...,0.0,0,False,True,False,False,False,False,False,False
3,76.03,0.0,0.0,0,391.0,400.0,294.0,200.0,750.0,500.0,...,0.0,0,False,True,False,False,False,False,False,False
4,174.97,1.0,0.0,0,500.0,500.0,192.0,0.0,18350.0,10750.0,...,0.0,0,False,True,False,False,False,False,False,False


In [24]:
# Checking for missing values
print("Missing values:", df.isnull().sum().sum())

Missing values: 0


In [25]:
# Splitting the data (X being the features, y being the prediction which team winning the round)
y = df["round_winner"]
X = df.drop("round_winner", axis=1) # Dropping a column instead of a row

In [26]:
# Split into 80% training 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.20, random_state=42)

scale = StandardScaler()


# Scale the data using Z-score normalization ((x - Mean / Standard Deviation))
X_train_scaled = scale.fit_transform(X_train)
X_test_scaled = scale.fit_transform(X_test)

# print(f"Training data shape: {X_train_scaled.shape}")
# print(f"Testing data shape: {X_test_scaled.shape}")

In [27]:
# Creating the logistic regression model and fitting the scaled data
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

# Testing the model
accuracy = model.score(X_test_scaled, y_test)

print(f"Model Accuracy: {accuracy * 100:.2f}")

Model Accuracy: 75.01


In [28]:
# Extract the weights (coefficients) and the feature names
weights = model.coef_[0]
feature_names = X.columns



# Put them into a Pandas DataFrame for easy reading

weights_df = pd.DataFrame({
    'Feature': feature_names,
    'Weight': weights
})
# Sort them out from highest to lowest
weights_df = weights_df.sort_values(by='Weight', ascending=False)

In [29]:
        # Look at the Top 5 T-sided features and Top 5 CT-sided features
print("--- Top 5 Advantages for Terrorists (T = 1) ---")
print(weights_df.head(5).to_string(index=False))
print("\n--- Top 5 Advantages for Counter-Terrorists (CT = 0) ---")
print(weights_df.tail(5).to_string(index=False))

--- Top 5 Advantages for Terrorists (T = 1) ---
        Feature   Weight
  t_weapon_ak47 0.562193
 t_weapon_sg553 0.515905
       t_health 0.507872
        t_armor 0.490416
t_players_alive 0.441036

--- Top 5 Advantages for Counter-Terrorists (CT = 0) ---
         Feature    Weight
ct_players_alive -0.285864
   ct_weapon_awp -0.321341
        ct_armor -0.491459
  ct_weapon_m4a4 -0.511945
       ct_health -0.640607


In [30]:

# Predicting which side wins
predictions = model.predict(X_test_scaled)

# Creating a copy
results_df = X_test.copy()

# X: Predictions, y: Example outputs
results_df['Actual_Winner'] = y_test
results_df['Predicted_Winner'] = predictions

# Convert it back from 0s and 1s to "CT" & "T" for better readability
results_df['Actual_Winner'] = results_df['Actual_Winner'].map({0: 'CT', 1: 'T'})
results_df['Predicted_Winner'] = results_df['Predicted_Winner'].map({0: 'CT', 1: 'T'})

columns_to_view = np.array(['ct_health', 't_health',
    'ct_armor', 't_armor',
    'ct_money', 't_money',
    'Actual_Winner', 'Predicted_Winner',
    "time_left",
    "ct_score", "t_score"])

print(results_df[columns_to_view].head(15))

        ct_health  t_health  ct_armor  t_armor  ct_money  t_money  \
47053       500.0     500.0     400.0    300.0     700.0    550.0   
28740       400.0     450.0     400.0    477.0   10300.0   6900.0   
92746       366.0     100.0     395.0     98.0    2350.0   2600.0   
60470       400.0     446.0     399.0    434.0     300.0  27000.0   
42953       100.0     200.0     100.0    200.0   10100.0  20950.0   
97836       400.0     500.0     400.0    500.0     150.0    800.0   
13483       142.0     170.0     167.0    100.0     900.0    500.0   
4232        190.0     288.0      96.0    345.0    1200.0  20350.0   
21597       374.0     400.0     397.0    400.0   17200.0   1800.0   
90658       368.0     234.0     400.0    277.0   15200.0   5700.0   
108947      500.0     419.0     440.0    477.0    5500.0   1550.0   
44210       162.0       9.0     184.0     96.0    2300.0    350.0   
106054      300.0     459.0     282.0    486.0    4400.0   3150.0   
37244       500.0     500.0     28